# Model Process: 
## Train + Prediction + Evaluation + Overlay

`yolo11` 모델을 `laboro little 3cls` 데이터셋에 대해 학습

흐름:
1. 학습 대상 config를 지정한다.
2. `config/model` + `config/dataset`을 runtime config로 합친다.
3. `run_train()`으로 YOLO 모델 학습 실행
4. `run_predict()`로 학습된 모델(`best.pt`)을 사용하여 test 이미지에 대한 예측 수행
    - YOLO label 형식으로 예측 결과 저장
5.  예측된 바운딩 박스를 원본 이미지 위에 시각화

In [1]:
import os
import sys

from pathlib import Path
from pprint import pprint

import torch

# Notebook에서는 직접 project root를 잡아 import 경로를 맞춘다.
candidate_roots = [Path.cwd().resolve(), Path.cwd().resolve().parent, Path('/home/hyeonjin/detect-and-reason')]
PROJECT_ROOT = next(
    path for path in candidate_roots
    if (path / 'pyproject.toml').exists() and (path / 'src').exists()
)
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('PROJECT_ROOT =', PROJECT_ROOT)

from src.config_loader import load_yaml, resolve_runtime_config

PROJECT_ROOT = /home/hyeonjin/detect-and-reason


### 학습 설정

In [2]:
# 실험 조합
MODEL_REF = 'yolo11_3cls'
DATASET_REF = 'little'

TRAIN_EPOCHS = 10
PREDICT_SOURCE = 'test'
EVAL_SPLIT = 'test'

RESULT_ROOT = PROJECT_ROOT / 'result' / 'detection_only' / DATASET_REF
PREDICTION_OUTPUT_DIR = RESULT_ROOT / f'{MODEL_REF}_prediction'
EVALUATION_OUTPUT_DIR = RESULT_ROOT / f'{MODEL_REF}_evaluation'
OVERLAY_OUTPUT_DIR = RESULT_ROOT / f'{MODEL_REF}_overlay'

print('MODEL_REF =', MODEL_REF)
print('DATASET_REF =', DATASET_REF)
print('TRAIN_EPOCHS =', TRAIN_EPOCHS)
print('PREDICTION_OUTPUT_DIR =', PREDICTION_OUTPUT_DIR)
print('EVALUATION_OUTPUT_DIR =', EVALUATION_OUTPUT_DIR)
print('OVERLAY_OUTPUT_DIR =', OVERLAY_OUTPUT_DIR)

MODEL_REF = yolo11_3cls
DATASET_REF = little
TRAIN_EPOCHS = 10
PREDICTION_OUTPUT_DIR = /home/hyeonjin/detect-and-reason/result/detection_only/little/yolo11_3cls_prediction
EVALUATION_OUTPUT_DIR = /home/hyeonjin/detect-and-reason/result/detection_only/little/yolo11_3cls_evaluation
OVERLAY_OUTPUT_DIR = /home/hyeonjin/detect-and-reason/result/detection_only/little/yolo11_3cls_overlay


In [3]:
# 학습 설정

print('torch.cuda.is_available():', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device_count:', torch.cuda.device_count())
    print('current_device:', torch.cuda.current_device())
    print('device_name:', torch.cuda.get_device_name(torch.cuda.current_device()))

runtime_cfg = resolve_runtime_config(
    model_ref=MODEL_REF,
    dataset_ref=DATASET_REF,
    stage='train',
)

print('[runtime summary]')
print('model_name:', runtime_cfg['model_name'])
print('family:', runtime_cfg['family'])
print('class_mode:', runtime_cfg['class_mode'])
print('dataset:', runtime_cfg['dataset']['key'])
print('data_yaml:', runtime_cfg['dataset']['data_yaml'])
print('train_dir:', runtime_cfg['dataset']['train_dir'])
print('val_dir:', runtime_cfg['dataset']['val_dir'])
print('test_dir:', runtime_cfg['dataset']['test_dir'])
print('runs_dir:', runtime_cfg['paths']['runs_dir'])
print('weights:', runtime_cfg['paths']['weights'])

train_preview = {
    'epochs': runtime_cfg['train'].get('epochs'),
    'imgsz': runtime_cfg['train'].get('imgsz'),
    'batch': runtime_cfg['train'].get('batch'),
    'workers': runtime_cfg['train'].get('workers'),
    'optimizer': runtime_cfg['train'].get('optimizer'),
    'lr0': runtime_cfg['train'].get('lr0'),
    'lrf': runtime_cfg['train'].get('lrf'),
    'cos_lr': runtime_cfg['train'].get('cos_lr'),
    'cache': runtime_cfg['dataset'].get('cache'),
    'augmentation': runtime_cfg['train'].get('augmentation', {}),
}
pprint(train_preview)

runtime_cfg['train']['epochs'] = TRAIN_EPOCHS

torch.cuda.is_available(): True
device_count: 7
current_device: 0
device_name: NVIDIA RTX PRO 6000 Blackwell Max-Q Workstation Edition
[runtime summary]
model_name: yolo11_3cls
family: yolo11
class_mode: 3cls
dataset: little
data_yaml: /home/hyeonjin/detect-and-reason/data/yolo/laboro_little_3cls/data.yaml
train_dir: /home/hyeonjin/detect-and-reason/data/yolo/laboro_little_3cls/train
val_dir: /home/hyeonjin/detect-and-reason/data/yolo/laboro_little_3cls/val
test_dir: /home/hyeonjin/detect-and-reason/data/yolo/laboro_little_3cls/test
runs_dir: /home/hyeonjin/detect-and-reason/runs/little/yolo11_3cls
weights: /home/hyeonjin/detect-and-reason/yolo11l.pt
{'augmentation': {'close_mosaic': 10,
                  'degrees': 10,
                  'fliplr': 0.5,
                  'hsv_h': 0.015,
                  'hsv_s': 0.2,
                  'hsv_v': 0.2,
                  'mixup': 0.1,
                  'mosaic': 0.2,
                  'scale': 0.2,
                  'translate': 0.1},
 'bat

### Train

In [6]:
from src.model.train import run_train

In [ ]:
# 학습 실행
train_results = run_train(runtime_cfg)
train_results

In [4]:
# 학습 결과
runs_dir = Path(runtime_cfg['paths']['runs_dir'])
weights_dir = runs_dir / 'weights'
best_weight = weights_dir / 'best.pt'
last_weight = weights_dir / 'last.pt'
results_csv = runs_dir / 'results.csv'

print('runs_dir:', runs_dir)
print('weights_dir exists:', weights_dir.exists())
print('best.pt exists:', best_weight.exists(), best_weight)
print('last.pt exists:', last_weight.exists(), last_weight)
print('results.csv exists:', results_csv.exists(), results_csv)

if not best_weight.exists():
    raise FileNotFoundError(f'best.pt not found: {best_weight}')

runs_dir: /home/hyeonjin/detect-and-reason/runs/little/yolo11_3cls
weights_dir exists: True
best.pt exists: True /home/hyeonjin/detect-and-reason/runs/little/yolo11_3cls/weights/best.pt
last.pt exists: True /home/hyeonjin/detect-and-reason/runs/little/yolo11_3cls/weights/last.pt
results.csv exists: True /home/hyeonjin/detect-and-reason/runs/little/yolo11_3cls/results.csv


### Prediction

In [5]:
from src.model.predict import run_predict
from src.evaluation.yolo_label_io import resolve_prediction_labels_dir

In [15]:
# prediction
predict_runtime_cfg = resolve_runtime_config(
    model_ref=MODEL_REF,
    dataset_ref=DATASET_REF,
    stage='predict',
    weights=str(best_weight),
    source=PREDICT_SOURCE,
)

# prediction 폴더에는 txt 라벨만 저장
# Ultralytics 기본 예측 이미지는 저장하지 않음
predict_runtime_cfg['paths']['prediction_dir'] = PREDICTION_OUTPUT_DIR
predict_runtime_cfg['predict']['save'] = False
predict_runtime_cfg['predict']['save_txt'] = True
predict_runtime_cfg['predict']['save_conf'] = True

print('[predict summary]')
print('weights:', predict_runtime_cfg['paths']['weights'])
print('source:', predict_runtime_cfg['predict']['resolved_source'])
print('prediction_dir:', predict_runtime_cfg['paths']['prediction_dir'])
print('save_images:', predict_runtime_cfg['predict']['save'])

predict_results = run_predict(predict_runtime_cfg)

pred_labels_dir = resolve_prediction_labels_dir(PREDICTION_OUTPUT_DIR)
print('pred_labels_dir:', pred_labels_dir)
predict_results

[predict summary]
weights: /home/hyeonjin/detect-and-reason/runs/little/yolo11_3cls/weights/best.pt
source: /home/hyeonjin/detect-and-reason/data/yolo/laboro_little_3cls/test/images
prediction_dir: /home/hyeonjin/detect-and-reason/result/detection_only/little/yolo11_3cls_prediction
save_images: False

image 1/73 /home/hyeonjin/detect-and-reason/data/yolo/laboro_little_3cls/test/images/IMG_1110.jpg: 640x480 5 half ripened tomatos, 20 green tomatos, 13.4ms
image 2/73 /home/hyeonjin/detect-and-reason/data/yolo/laboro_little_3cls/test/images/IMG_1112.jpg: 640x480 1 half ripened tomato, 35 green tomatos, 13.0ms
image 3/73 /home/hyeonjin/detect-and-reason/data/yolo/laboro_little_3cls/test/images/IMG_1113.jpg: 640x480 2 fully ripened tomatos, 9 half ripened tomatos, 50 green tomatos, 12.6ms
image 4/73 /home/hyeonjin/detect-and-reason/data/yolo/laboro_little_3cls/test/images/IMG_1116.jpg: 640x480 1 fully ripened tomato, 3 half ripened tomatos, 30 green tomatos, 12.6ms
image 5/73 /home/hyeonjin

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'fully ripened tomato', 1: 'half ripened tomato', 2: 'green tomato'}
 obb: None
 orig_img: array([[[ 76, 120, 113],
         [ 81, 125, 118],
         [ 87, 129, 122],
         ...,
         [ 81, 132, 124],
         [ 78, 129, 121],
         [ 80, 131, 123]],
 
        [[ 63, 107, 100],
         [ 54,  99,  90],
         [ 50,  92,  85],
         ...,
         [ 71, 122, 114],
         [ 71, 122, 114],
         [ 72, 123, 115]],
 
        [[ 64, 107,  98],
         [ 52,  95,  84],
         [ 43,  84,  76],
         ...,
         [ 66, 117, 110],
         [ 62, 114, 107],
         [ 59, 111, 104]],
 
        ...,
 
        [[177, 204, 225],
         [205, 232, 253],
         [158, 185, 205],
         ...,
         [175, 188, 202],
         [ 84,  97, 111],
         [ 57,  70,  84]],
 
        [[210, 236, 253],
         [224, 250, 255],

### Evaluation

In [6]:
from src.evaluation import (
    evaluate_yolo_label_predictions,
    save_detection_evaluation_artifacts,
)
from src.evaluation.yolo_label_io import (
    resolve_prediction_labels_dir,
    resolve_yolo_split_dirs,
)

In [7]:
eval_runtime_cfg = resolve_runtime_config(
    model_ref=MODEL_REF,
    dataset_ref=DATASET_REF,
    stage='predict',
    weights=str(best_weight),
    source=EVAL_SPLIT,
)

gt_images_dir, gt_labels_dir = resolve_yolo_split_dirs(
    eval_runtime_cfg['dataset'][f'{EVAL_SPLIT}_dir']
)

pred_labels_dir = resolve_prediction_labels_dir(PREDICTION_OUTPUT_DIR)
class_names = eval_runtime_cfg['class_names'] or eval_runtime_cfg['dataset']['class_names']

eval_results = evaluate_yolo_label_predictions(
    gt_images_dir=gt_images_dir,
    gt_labels_dir=gt_labels_dir,
    pred_labels_dir=pred_labels_dir,
    class_names=class_names,
    iou_threshold=eval_runtime_cfg['predict'].get('resolved_iou') or 0.5,
    weight_reference=eval_runtime_cfg['paths']['weights'],
    model_imgsz=int(eval_runtime_cfg.get('train', {}).get('imgsz', 640)),
)

artifact_paths = save_detection_evaluation_artifacts(
    eval_results,
    EVALUATION_OUTPUT_DIR,
)

stats = eval_results['detailed_statistics']['total_statistics']
det_metrics = eval_results['detection_metrics']

print('[evaluation summary]')
print('gt_labels_dir   :', gt_labels_dir)
print('pred_labels_dir :', pred_labels_dir)
print('output_dir      :', EVALUATION_OUTPUT_DIR)
print(f"precision         : {stats['overall_precision']:.4f}")
print(f"recall            : {stats['overall_recall']:.4f}")
print(f"f1                : {stats['overall_f1']:.4f}")
print(f"detection_acc     : {stats['detection_acc']:.4f}")
print(f"classification_acc: {stats['classification_acc']:.4f}")
print(f"overall_acc       : {stats['overall_acc']:.4f}")
print(f"mAP@0.5           : {det_metrics['map_50']:.4f}")
print(f"mAP@0.5:0.95      : {det_metrics['map']:.4f}")
print(f"mAP@0.75          : {det_metrics['map_75']:.4f}")
print(f"CA-mAP@0.5        : {det_metrics['ca_map_50']:.4f}")
print(f"CA-mAP@0.5:0.95   : {det_metrics['ca_map']:.4f}")

pprint(artifact_paths)

/home/hyeonjin/detect-and-reason/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[evaluation summary]
gt_labels_dir   : /home/hyeonjin/detect-and-reason/data/yolo/laboro_little_3cls/test/labels
pred_labels_dir : /home/hyeonjin/detect-and-reason/result/detection_only/little/yolo11_3cls_prediction/labels
output_dir      : /home/hyeonjin/detect-and-reason/result/detection_only/little/yolo11_3cls_evaluation
precision         : 0.7363
recall            : 0.8224
f1                : 0.7770
detection_acc     : 0.8687
classification_acc: 0.9468
overall_acc       : 0.8224
mAP@0.5           : 0.7595
mAP@0.5:0.95      : 0.6078
mAP@0.75          : 0.7035
CA-mAP@0.5        : 0.8310
CA-mAP@0.5:0.95   : 0.6552
{'confusion_csv': PosixPath('/home/hyeonjin/detect-and-reason/result/detection_only/little/yolo11_3cls_evaluation/confusion_matrix.csv'),
 'json': PosixPath('/home/hyeonjin/detect-and-reason/result/detection_only/little/yolo11_3cls_evaluation/evaluation_results.json'),
 'per_class_csv': PosixPath('/home/hyeonjin/detect-and-reason/result/detection_only/little/yolo11_3cls_eval

### Overlay

In [9]:
from src.visualization import save_yolo_label_overlays

In [10]:
overlay_runtime_cfg = resolve_runtime_config(
    model_ref=MODEL_REF,
    dataset_ref=DATASET_REF,
    stage='predict',
    weights=str(best_weight),
    source=EVAL_SPLIT,
)

gt_images_dir, _ = resolve_yolo_split_dirs(
    overlay_runtime_cfg['dataset'][f'{EVAL_SPLIT}_dir']
)

pred_labels_dir = resolve_prediction_labels_dir(PREDICTION_OUTPUT_DIR)
class_names = overlay_runtime_cfg['class_names'] or overlay_runtime_cfg['dataset']['class_names']

overlay_cfg_path = PROJECT_ROOT / 'config' / 'visualization' / 'detection_overlay.yaml'
overlay_cfg = load_yaml(overlay_cfg_path)

rendered = save_yolo_label_overlays(
    labels_dir=pred_labels_dir,
    img_root=gt_images_dir,
    output_dir=OVERLAY_OUTPUT_DIR,
    class_names=class_names,
    overlay_config=overlay_cfg,
)

print('[overlay summary]')
print('pred_labels_dir :', pred_labels_dir)
print('img_root        :', gt_images_dir)
print('overlay_config  :', overlay_cfg_path)
print('output_dir      :', OVERLAY_OUTPUT_DIR)
print('rendered        :', rendered)

[overlay summary]
pred_labels_dir : /home/hyeonjin/detect-and-reason/result/detection_only/little/yolo11_3cls_prediction/labels
img_root        : /home/hyeonjin/detect-and-reason/data/yolo/laboro_little_3cls/test/images
overlay_config  : /home/hyeonjin/detect-and-reason/config/visualization/detection_overlay.yaml
output_dir      : /home/hyeonjin/detect-and-reason/result/detection_only/little/yolo11_3cls_overlay
rendered        : 73
